First, let's create a Python file `streamlit_app.py` that will contain our Streamlit application code. This app will allow users to input passenger details and get a survival prediction.

In [ ]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd

st.title('Titanic Survival Prediction')
st.write('Enter passenger details to predict survival.')

# Input fields for features
pclass = st.selectbox('Passenger Class', [1, 2, 3])
sex = st.selectbox('Sex', ['male', 'female'])
age = st.slider('Age', 0, 80, 30)
sibsp = st.slider('Number of Siblings/Spouses Aboard', 0, 8, 0)
parch = st.slider('Number of Parents/Children Aboard', 0, 6, 0)
fare = st.slider('Fare', 0.0, 500.0, 30.0)
embarked = st.selectbox('Port of Embarkation', ['C', 'Q', 'S'])

# Create a DataFrame from inputs
input_data = pd.DataFrame({
    'Pclass': [pclass],
    'Sex': [1 if sex == 'male' else 0], # Assuming male=1, female=0 after encoding
    'Age': [age],
    'SibSp': [sibsp],
    'Parch': [parch],
    'Fare': [fare],
    'Embarked_C': [1 if embarked == 'C' else 0],
    'Embarked_Q': [1 if embarked == 'Q' else 0],
    'Embarked_S': [1 if embarked == 'S' else 0]
})

# --- Placeholder for Model Prediction ---
st.subheader('Prediction Results')

if st.button('Predict Survival'):
    # In a real scenario, you would load and use your trained model here.
    # For demonstration, we'll use a very simple rule-based 'model'.
    # Replace this with your actual model loading and prediction logic.
    # Example: model = joblib.load('your_model.pkl')
    # prediction = model.predict(input_data_processed)

    # Simple rule-based prediction for demonstration
    if pclass == 1 and sex == 'female':
        prediction = 1 # High chance of survival
    elif pclass == 3 and sex == 'male' and age > 40:
        prediction = 0 # Low chance of survival
    elif pclass == 3 and sex == 'male' and age < 10:
        prediction = 1 # Child male in third class might survive
    elif age <= 10:
        prediction = 1 # Children often prioritized
    elif fare > 100:
        prediction = 1 # High fare usually means better class and survival chance
    else:
        prediction = 0 # Default to not surviving for other cases

    if prediction == 1:
        st.success('Prediction: Survived! 🚢')
    else:
        st.error('Prediction: Did Not Survive. 🌊')

    st.write('---')
    st.write('**Input Data:**')
    st.write(input_data)



Writing streamlit_app.py


In [ ]:
from pyngrok import ngrok, conf
import subprocess
import os
import time

# Terminate any existing ngrok tunnels
ngrok.kill()

# Set ngrok authtoken
conf.get_default().auth_token = ngrok_key

# Start Streamlit in the background
# We use nohup to detach the process from the current shell
# and redirect output to a log file to prevent Colab from stopping it.
# The `--server.port` must match the port ngrok will tunnel.
streamlit_process = subprocess.Popen(
    ['nohup', 'streamlit', 'run', 'streamlit_app.py', '--server.port', '8501', '&'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    preexec_fn=os.setpgrp # Detach from parent process group
)

# Wait a moment for Streamlit to start
time.sleep(5)

# Open a ngrok tunnel to the Streamlit port
try:
    public_url = ngrok.connect(8501)
    print(f'Your Streamlit app is live at: {public_url}')
    print('Click the link above to access your app.')
except Exception as e:
    print(f"Error creating ngrok tunnel: {e}")
    print("Please ensure your ngrok key is valid and you have internet access.")
    if streamlit_process.poll() is not None:
        print("Streamlit process exited. Check logs:")
        print(f"stdout: {streamlit_process.stdout.read().decode()}")
        print(f"stderr: {streamlit_process.stderr.read().decode()}")


Your Streamlit app is live at: NgrokTunnel: "https://polygon-semifinal-glance.ngrok-free.dev" -> "http://localhost:8501"
Click the link above to access your app.


In [ ]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt